# Step 3 & 4: CNN Training on HAM10000 — ResNet50 & EfficientNet-B0

Trains and evaluates ResNet50 (baseline) and EfficientNet-B0 (comparison) on HAM10000.
**Run on Google Colab with a T4 GPU.**

> ResNet50 is the baseline. EfficientNet-B0 is trained as a lighter comparison model.

---

## Setup — Project Files

**Option A:** Clone from GitHub:
```bash
# !git clone https://github.com/raomateen12/Skin_cancer.git
# %cd Skin_cancer
```

**Option B:** Upload project zip to Colab:
```python
# from google.colab import files
# files.upload()  # upload fair-medical-ai.zip
# !unzip fair-medical-ai.zip
# %cd fair-medical-ai
```

In [ ]:
# Cell 1: Check GPU
import torch
print('CUDA available:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

In [ ]:
# Cell 2: Install dependencies
!pip install kaggle albumentations grad-cam shap langchain langchain-community \
    langchain-huggingface sentence-transformers faiss-cpu pypdf transformers accelerate -q

## Kaggle Credentials

> **Important:** Do NOT print or share your `kaggle.json` contents.
> The cell below only uploads the file — it does not display credentials.

In [ ]:
# Cell 3: Upload kaggle.json securely
from google.colab import files
print('Upload your kaggle.json (do not share its contents).')
uploaded = files.upload()

In [ ]:
# Cell 4: Move kaggle.json to correct location
import os
os.makedirs('/root/.kaggle', exist_ok=True)
!mv kaggle.json /root/.kaggle/kaggle.json
!chmod 600 /root/.kaggle/kaggle.json
print('Kaggle credentials configured.')

In [ ]:
# Cell 5: Download HAM10000 dataset (~5 GB)
!python -m src.download_kaggle_dataset

In [ ]:
# Cell 6: Prepare data — build train/val/test CSVs
!python -m src.prepare_data

In [ ]:
# Cell 7: Verify DataLoaders
!python -m src.check_dataloaders

In [ ]:
# Cell 8: Train ResNet50 (baseline)
# Saves best checkpoint to checkpoints/best_resnet50.pth
!python -m src.train --model_name resnet50

In [ ]:
# Cell 9: Evaluate ResNet50 on test set
!python -m src.evaluate --model_name resnet50

In [ ]:
# Cell 10: Train EfficientNet-B0 (comparison)
# Saves best checkpoint to checkpoints/best_efficientnet_b0.pth
!python -m src.train --model_name efficientnet_b0

In [ ]:
# Cell 11: Evaluate EfficientNet-B0 on test set
!python -m src.evaluate --model_name efficientnet_b0

In [ ]:
# Cell 12: Compare models — prints table, saves CSV and bar chart
!python -m src.compare_models

In [ ]:
# Cell 13: Display all result plots
from IPython.display import Image, display
from pathlib import Path

plots = [
    ('ResNet50 training curves',     'results/resnet50_training_curves.png'),
    ('ResNet50 confusion matrix',    'results/resnet50_confusion_matrix.png'),
    ('EfficientNet-B0 training',     'results/efficientnet_b0_training_curves.png'),
    ('EfficientNet-B0 confusion',    'results/efficientnet_b0_confusion_matrix.png'),
    ('Model comparison bar chart',   'results/model_comparison_bar.png'),
]

for title, path in plots:
    if Path(path).exists():
        print(f'\n{title}:')
        display(Image(path))
    else:
        print(f'[not found] {path}')